In [1]:
import json
import re
import os
from PIL import Image
import pdfplumber
from transformers import pipeline
import torch
import cv2
import numpy as np

# from pipe_fn import pipe
from transformers import pipeline
from output_utils import save_split_output


from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)


# =========================
# CROP TABLES
# =========================
def crop_claim_tables(pdf_path, output_dir="cropped_tables_sunlife"):
    os.makedirs(output_dir, exist_ok=True)
    cropped_images = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            print(f"\n📄 Processing Page {page_num}")

            claim_hits = page.search("Claim #:")
            if not claim_hits:
                claim_hits = page.search("Claim #")

            total_hits = page.search("TOTALS")

            if not claim_hits or not total_hits:
                continue

            table_count = min(len(claim_hits), len(total_hits))

            for idx in range(table_count):
                start_y = claim_hits[idx]["top"] - 5
                end_y   = total_hits[idx]["bottom"] + 10

                if start_y >= end_y:
                    continue

                bbox = (0, start_y, page.width, end_y)
                cropped_page = page.crop(bbox)

                image_path = os.path.join(
                    output_dir,
                    f"page_{page_num}_table_{idx+1}.png"
                )
                cropped_page.to_image(resolution=500).save(image_path)
                print(f"✅ Saved: {image_path}")

                expected_rows = count_service_rows(page, start_y, end_y)

                cropped_images.append({
                    "page":          page_num,
                    "table":         idx + 1,
                    "image_path":    image_path,
                    "expected_rows": expected_rows,
                })

    return cropped_images


def count_service_rows(page, region_top, region_bottom):
    words = page.extract_words()
    service_rows = set()
    for w in words:
        text = w["text"].strip()
        if re.fullmatch(r"D\d{4}", text):
            y = float(w["top"])
            if region_top <= y <= region_bottom:
                service_rows.add(round(y, 1))
    return len(service_rows)


# =========================
# TABLE ENHANCEMENT
# =========================
def make_table(image_path):
    img  = cv2.imread(image_path)
    gray = cv2.imread(image_path, 0)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
    sums  = np.sum(thresh, axis=1)
    th    = (thresh.shape[1] * 255) * 0.6
    lines = np.where(sums > th)[0]
    for l in lines:
        cv2.line(img, (0, l), (thresh.shape[1], l), (0, 0, 0), 1)
    return img


def convert_amounts_to_string(obj):
    amount_fields = {
        "submitted_charges", "allowed_amount", "deductible",
        "ppo_savings", "patient_responsibility", "plan_payment",
    }
    if isinstance(obj, dict):
        new_obj = {}
        for k, v in obj.items():
            if k in amount_fields:
                try:
                    new_obj[k] = f"{float(str(v).replace('$','').replace(',','').strip()):.2f}"
                except Exception:
                    new_obj[k] = ""
            else:
                new_obj[k] = convert_amounts_to_string(v)
        return new_obj
    elif isinstance(obj, list):
        return [convert_amounts_to_string(i) for i in obj]
    else:
        return obj


# =========================
# PROMPT  — uses pdf_name as eob_id
# =========================
def build_prompt(pdf_name: str) -> str:
    return f"""
Extract structured data from this dental EOB claim table.

STRICT RULES

1. Extract ONLY:
   - patient_name   (value after "Patient:")
   - provider       (Value after "Provider:")
   - relationship   (value after "Relationship:")

   from the header above the table.

2. Extract ALL service rows exactly as shown.

3. Preserve row order.

4. Do NOT skip duplicate rows.

5. Read the table strictly LEFT to RIGHT.

6. Every service object must correspond to ONE visible table row.

7. Never merge two rows.

8. Never create rows that do not exist.

9. Never use TOTALS values inside service rows.

10. Stop reading service rows when the row labeled "TOTALS" is reached.

11. Extract the TOTALS row separately into the "totals" object.

12. Procedure code must match D followed by exactly 4 digits.
    Examples: D1110  D0220  D4341

13. Ignore: Tooth column, Remark Codes, Submitted Service Description,
    Allowed Service, Co-Pay %.

14. Money values: remove "$" and commas, keep decimals, return as strings.
    Example: "$1,234.00" -> "1234.00"

15. If a field is blank in the table return "".

16. Never infer missing values.

17. Return ONLY valid JSON — no markdown, no explanation, no comments.

18. If Patient Resp equals Allowed Amount, extract both values exactly
    as displayed.

19. Deductible must be extracted from the visible Deductible column for EACH row.

20. Do NOT copy deductible values from neighboring columns.

21. If a non-zero deductible appears in a service row, extract that exact value.

22. Totals validation:
    Sum(service.deductible) MUST equal totals.deductible
    whenever all deductible values are visible.

23. Read columns using the table header alignment only.
    Never infer values based on nearby columns.

24. PPO Savings and Deductible are separate columns.
    Never shift values between them.

25. Patient Responsibility and Plan Payment are separate columns.
    Never shift values between them.

COLUMN MAPPING

service_date           <- Service Date
procedure_code         <- Submitted Services
submitted_charges      <- Submitted Charges
allowed_amount         <- Allowed Amount
deductible             <- Deductible
ppo_savings            <- PPO Savings
patient_responsibility <- Patient Resp
plan_payment           <- Plan Payment

OUTPUT JSON SCHEMA
OUTPUT JSON SCHEMA

{{

  "patient_name": {{
    "value": "",
    "confidence": 0.0
  }},

  "relationship": {{
    "value": "",
    "confidence": 0.0
  }},

  "provider": {{
    "value": "",
    "confidence": 0.0
  }},

  "dob": {{
    "value": "",
    "confidence": 0.0
  }},

  "services": [
    {{

      "service_date": {{
        "value": "",
        "confidence": 0.0
      }},

      "procedure_code": {{
        "value": "",
        "confidence": 0.0
      }},

      "submitted_charges": {{
        "value": "",
        "confidence": 0.0
      }},

      "allowed_amount": {{
        "value": "",
        "confidence": 0.0
      }},

      "deductible": {{
        "value": "",
        "confidence": 0.0
      }},

      "ppo_savings": {{
        "value": "",
        "confidence": 0.0
      }},

      "patient_responsibility": {{
        "value": "",
        "confidence": 0.0
      }},

      "plan_payment": {{
        "value": "",
        "confidence": 0.0
      }}

    }}
  ],

  "totals": {{

    "submitted_charges": {{
      "value": "",
      "confidence": 0.0
    }},

    "allowed_amount": {{
      "value": "",
      "confidence": 0.0
    }},

    "deductible": {{
      "value": "",
      "confidence": 0.0
    }},

    "ppo_savings": {{
      "value": "",
      "confidence": 0.0
    }},

    "patient_responsibility": {{
      "value": "",
      "confidence": 0.0
    }},

    "plan_payment": {{
      "value": "",
      "confidence": 0.0
    }}

  }}

}}

 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{{
  "value": "",
  "confidence": ""
}}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.

VALIDATION RULES
- Number of service objects must equal the number of visible service rows.
- Duplicate procedure codes must be extracted as separate rows.
- Do not include the TOTALS row inside services.
- Output ONLY JSON.
"""


# =========================
# AMOUNT HELPERS
# =========================
def parse_amount(x) -> float:
    if x is None or str(x).strip() == "":
        return 0.0
    try:
        return float(str(x).replace("$", "").replace(",", "").strip())
    except ValueError:
        return 0.0


def compute_totals_from_services(services: list) -> dict:
    fields = [
        "submitted_charges", "allowed_amount", "deductible",
        "ppo_savings", "patient_responsibility",  "plan_payment",
    ]
    return {
        f: round(sum(parse_amount(s.get(f, "")) for s in services), 2)
        for f in fields
    }


def services_are_empty(services: list) -> bool:
    """Return True when every amount field in every service row is blank/zero."""
    amount_fields = [
        "submitted_charges", "allowed_amount", "deductible",
        "ppo_savings", "patient_responsibility", "plan_payment",
    ]
    for svc in services:
        for f in amount_fields:
            if str(svc.get(f, "")).strip() not in ("", "0.00", "0"):
                return False
    return True


# =========================
# VALIDATION
# =========================
def validate_patient_totals(patient: dict, patient_name: str, expected_row_count: int):
    services = patient.get("services", [])
    totals   = patient.get("totals", {})

    field_names = list(compute_totals_from_services([]).keys())   # ADD
    total_fields = len(field_names)    

    if not services:
        field_errors = [                                            # ADD
            {"field": f, "computed": 0.0, "extracted": None}
            for f in field_names
        ]
        return False, "No services found", [{"error": "empty services"}] + field_errors, total_fields

    if services_are_empty(services):
        msg = "All service rows are empty — model likely failed to extract data"
        print(f"\n❌ {msg}")
        field_errors = [                                            # ADD
            {"field": f, "computed": 0.0, "extracted": None}
            for f in field_names
        ]
        return False, msg, [{"error": "all_service_rows_empty"}] + field_errors, total_fields

    computed_totals = compute_totals_from_services(services)
    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [{patient_name}]")
    print("-" * 80)

    for field, computed_value in computed_totals.items():
        extracted_value = round(parse_amount(totals.get(field, "")), 2)
        diff  = round(computed_value - extracted_value, 2)
        match = abs(diff) <= 0.01
        icon   = "✅" if match else "❌"
        status = "MATCH" if match else "MISMATCH"

        if not match:
            has_error = True
            errors.append({
                "type":       "field_mismatch",
                "field":      field,
                "computed":   computed_value,
                "extracted":  extracted_value,
                "difference": diff,
            })

        line = (
            f"{icon} {field:25s} computed={computed_value:<10} "
            f"| extracted={extracted_value:<10} {status}"
        )
        print(line)
        result_validation += "\n" + line

    extracted_row_count = len(services)
    match  = expected_row_count == extracted_row_count
    icon   = "✅" if match else "❌"
    status = "MATCH" if match else "MISMATCH"

    if not match:
        has_error = True
        errors.append({
            "type":           "row_count_mismatch",
            "expected_rows":  expected_row_count,
            "extracted_rows": extracted_row_count,
        })

    line = (
        f"{icon} {'total_record_rows':25s} computed={expected_row_count:<10} "
        f"| extracted={extracted_row_count:<10} {status}"
    )
    print(line)
    result_validation += "\n" + line
    print("-" * 80)

    if has_error:
        print(f"❌ [{patient_name}] Validation FAILED\n")
        return False, result_validation, errors, total_fields 
    else:
        print(f"✅ [{patient_name}] Validation PASSED\n")
        return True, result_validation, [], total_fields 


# =========================
# JSON CLEANER
# =========================
def extract_json(text: str) -> dict:
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError("No JSON object found in model output")
    return json.loads(text[start:end])


def save_json(data, output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


# =========================
# DENIAL CHECK
# =========================
def check_claim_denied(pdf_path: str) -> str:
    denial_keywords = ["denied", "denial"]
    stop_phrase     = "Appealing a denial of the dental claim"

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            full_text = page.extract_text()
            if not full_text:
                continue
            searchable = full_text.lower()
            if stop_phrase.lower() in searchable:
                searchable = searchable.split(stop_phrase.lower())[0]
            for keyword in denial_keywords:
                if keyword in searchable:
                    print(f"claim denied keyword found: {keyword}")
                    return "denied"

    return "not denied"


# =========================
# RETRY HELPER
# =========================
MAX_RETRIES = 2

def run_model_with_retry(image: Image.Image, prompt: str, expected_rows: int):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text":  prompt},
            ],
        }
    ]

    last_parsed = None

    for attempt in range(1, MAX_RETRIES + 1):
        print(f"  🔄 Model attempt {attempt}/{MAX_RETRIES}")

        with torch.no_grad():
            output = pipe(
                messages,
                max_new_tokens=1500,
                temperature=0.0,
                do_sample=False,
            )

        raw = output[0]["generated_text"]
        if isinstance(raw, list):
            raw = raw[-1]["content"]

        try:
            parsed = extract_json(raw)
        except Exception as e:
            print(f"  ❌ JSON parse failed on attempt {attempt}: {e}")
            continue

        services = parsed.get("services", [])
        row_ok   = (len(services) == expected_rows)
        empty_ok = not services_are_empty(services)

        if row_ok and empty_ok:
            print(f"  ✅ Accepted on attempt {attempt}")
            return parsed

        print(
            f"  ⚠️  attempt {attempt}: rows={len(services)} (expected {expected_rows}), "
            f"all_empty={not empty_ok}"
        )
        last_parsed = parsed

    print(f"  ⚠️  All {MAX_RETRIES} attempts exhausted — using last result")
    return last_parsed


# =========================
# MAIN PIPELINE
# =========================
def run_pipeline(pdf_path: str, output_dir: str = "EOB_OUTPUT/SUNLIFE", company_name= "Sunlife"):
    # pdf_name is used as eob_id throughout
    pdf_name = os.path.basename(pdf_path).split(".")[0].split("_")[-1]
    pdf_full_name = os.path.basename(pdf_path)


    base_dir    = os.path.join(output_dir, pdf_name)
    cropped_dir = os.path.join(base_dir, "cropped_images")
    json_output_path = os.path.join(base_dir, f"{pdf_name}_output.json")

    if os.path.exists(json_output_path):
        print(f"⏭️  Skipping {pdf_name} — output already exists")
        return None

    os.makedirs(base_dir,    exist_ok=True)
    os.makedirs(cropped_dir, exist_ok=True)

    image_paths  = crop_claim_tables(pdf_path, output_dir=cropped_dir)
    is_denied    = check_claim_denied(pdf_path)
    print(f"claim status: {is_denied}")

    # Build prompt once with pdf_name as eob_id
    final_prompt = build_prompt(pdf_name)

    results = []

    for idx, item in enumerate(image_paths):
        img_path      = item["image_path"]
        expected_rows = item["expected_rows"]

        print(f"\nProcessing table {idx+1}/{len(image_paths)}")

        image_cv  = make_table(img_path)
        image_pil = Image.fromarray(image_cv).convert("RGB")

        parsed = run_model_with_retry(image_pil, final_prompt, expected_rows)

        if parsed is None:
            print(f"❌ Skipping table {idx+1} — model returned nothing usable")
            continue

        model_confidence = calculate_model_confidence(parsed)   # ADD
        parsed = _unwrap_vlm_output(parsed)                     # ADD
        parsed["_model_confidence"] = model_confidence 

        # Normalise relationship key spelling
        if "Realationship" in parsed:
            parsed["relationship"] = parsed.pop("Realationship")

        parsed = convert_amounts_to_string(parsed)
        parsed["_expected_rows"] = expected_rows

        print("Extracted:")
        print(json.dumps(parsed, indent=2))

        results.append(parsed)

    # Validate every patient
    for patient in results:
        is_valid, log, errors, total_fields  = validate_patient_totals(
            patient=patient,
            patient_name=patient.get("patient_name", "UNKNOWN"),
            expected_row_count=patient.get("_expected_rows", 0),
        )
        patient["validation"] = {
            "status": is_valid,
            "errors": errors,
        }
        patient["_total_fields"] = total_fields   # ADD

    confidence_results = list(results)                              # ADD
    confidence_score = calculate_eob_confidence(confidence_results)  # ADD

    for patient in results:                                          # ADD (cleanup)
        patient.pop("_expected_rows", None)
        patient.pop("_total_fields", None)
        patient.pop("_model_confidence", None)

    final = [
        {
            "eob_id":       pdf_name,
            "file_name":pdf_full_name,
            "claim_status": is_denied,
            "payor": "Sunlife",
            "confidence_score": confidence_score,  
            "patients":     results,
        }
    ]

    success_path, failed_path = save_split_output(
                            final,
                            company_name=company_name,
                            pdf_name=pdf_name,
                            pdf_path=pdf_path,
                            cropped_dir=cropped_dir,
                        )
                    
    print(f"\n📁 Cropped images : {cropped_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final

W0901 19:21:12.250000 3533314 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 19:21:12.264000 3533314 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Sunlife/Sunlife PDF's/Pmt_EOP_270721701.pdf")


📄 Processing Page 1
✅ Saved: EOB_OUTPUT/SUNLIFE/270721701/cropped_images/page_1_table_1.png

📄 Processing Page 2

📄 Processing Page 3


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


claim status: not denied

Processing table 1/1
  🔄 Model attempt 1/2


[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ Accepted on attempt 1
Extracted:
{
  "patient_name": "CARLOS FOSTER",
  "provider": "DUC TANG",
  "relationship": "Member",
  "dob": "",
  "services": [
    {
      "service_date": "08/11/22",
      "procedure_code": "D4910",
      "submitted_charges": "113.00",
      "allowed_amount": "113.00",
      "deductible": "0.00",
      "ppo_savings": "0.00",
      "patient_responsibility": "22.60",
      "plan_payment": "90.40"
    },
    {
      "service_date": "08/11/22",
      "procedure_code": "D0120",
      "submitted_charges": "40.00",
      "allowed_amount": "40.00",
      "deductible": "0.00",
      "ppo_savings": "0.00",
      "patient_responsibility": "0.00",
      "plan_payment": "40.00"
    }
  ],
  "totals": {
    "submitted_charges": "153.00",
    "allowed_amount": "153.00",
    "deductible": "0.00",
    "ppo_savings": "0.00",
    "patient_responsibility": "22.60",
    "plan_payment": "130.40"
  },
  "_model_confidence": 1.0,
  "_expected_rows": 2
}

🔍 Validation for [CARLOS

[{'eob_id': '270721701',
  'file_name': 'Pmt_EOP_270721701.pdf',
  'claim_status': 'not denied',
  'payor': 'Sunlife',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'CARLOS FOSTER',
    'provider': 'DUC TANG',
    'relationship': 'Member',
    'dob': '',
    'services': [{'service_date': '08/11/22',
      'procedure_code': 'D4910',
      'submitted_charges': '113.00',
      'allowed_amount': '113.00',
      'deductible': '0.00',
      'ppo_savings': '0.00',
      'patient_responsibility': '22.60',
      'plan_payment': '90.40'},
     {'service_date': '08/11/22',
      'procedure_code': 'D0120',
      'submitted_charges': '40.00',
      'allowed_amount': '40.00',
      'deductible': '0.00',
      'ppo_savings': '0.00',
      'patient_responsibility': '0.00',
      'plan_payment': '40.00'}],
    'totals': {'submitted_charges': '153.00',
     'allowed_amount': '153.00',
     'deductible': '0.00',
     'ppo_savings': '0.00',
     'patient_responsibility': '22.60',
     'plan

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Sunlife/Sunlife PDF's/Pmt_EOP_270721701.pdf")


📄 Processing Page 1
✅ Saved: EOB_OUTPUT/SUNLIFE/270721701/cropped_images/page_1_table_1.png

📄 Processing Page 2

📄 Processing Page 3


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


claim status: not denied

Processing table 1/1
  🔄 Model attempt 1/2


[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ Accepted on attempt 1
Extracted:
{
  "patient_name": "CARLOS FOSTER",
  "provider": "DUC TANG",
  "relationship": "Member",
  "dob": "",
  "services": [
    {
      "service_date": "08/11/22",
      "procedure_code": "D4910",
      "submitted_charges": "113.00",
      "allowed_amount": "113.00",
      "deductible": "0.00",
      "ppo_savings": "0.00",
      "patient_responsibility": "22.60",
      "plan_payment": "90.40"
    },
    {
      "service_date": "08/11/22",
      "procedure_code": "D0120",
      "submitted_charges": "40.00",
      "allowed_amount": "40.00",
      "deductible": "0.00",
      "ppo_savings": "0.00",
      "patient_responsibility": "0.00",
      "plan_payment": "40.00"
    }
  ],
  "totals": {
    "submitted_charges": "153.00",
    "allowed_amount": "153.00",
    "deductible": "0.00",
    "ppo_savings": "0.00",
    "patient_responsibility": "22.60",
    "plan_payment": "130.40"
  },
  "_model_confidence": 0.96,
  "_expected_rows": 2
}

🔍 Validation for [CARLO

[{'eob_id': '270721701',
  'claim_status': 'not denied',
  'payor': 'Sunlife',
  'confidence_score': 96.0,
  'patients': [{'patient_name': 'CARLOS FOSTER',
    'provider': 'DUC TANG',
    'relationship': 'Member',
    'dob': '',
    'services': [{'service_date': '08/11/22',
      'procedure_code': 'D4910',
      'submitted_charges': '113.00',
      'allowed_amount': '113.00',
      'deductible': '0.00',
      'ppo_savings': '0.00',
      'patient_responsibility': '22.60',
      'plan_payment': '90.40'},
     {'service_date': '08/11/22',
      'procedure_code': 'D0120',
      'submitted_charges': '40.00',
      'allowed_amount': '40.00',
      'deductible': '0.00',
      'ppo_savings': '0.00',
      'patient_responsibility': '0.00',
      'plan_payment': '40.00'}],
    'totals': {'submitted_charges': '153.00',
     'allowed_amount': '153.00',
     'deductible': '0.00',
     'ppo_savings': '0.00',
     'patient_responsibility': '22.60',
     'plan_payment': '130.40'},
    'validation': {